# Movies Dataset — Process Phase

Drop the 6 corrupted/shifted rows found in Prepare, coerce numeric types, drop duplicate `id`
rows, parse `release_date` and the stringified `genres` list, treat `budget`/`revenue` of `0` as
missing, and compute `roi`/`profit` only where both are meaningfully known (> $1,000). Source:
`data/raw/movies_metadata.csv` (not committed — see `data/raw/README.md`). Outputs:
`data/processed/movies_clean.parquet` (movie-level) and `data/processed/movie_genres.parquet`
(one row per movie x genre).

## Step 1 — Load raw data

In [1]:
import pandas as pd
import numpy as np
import ast
import os

RAW = "../data/raw"
OUT_DIR = "../data/processed"
SUMMARY_DIR = "../data/summary"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(SUMMARY_DIR, exist_ok=True)

df = pd.read_csv(os.path.join(RAW, "movies_metadata.csv"), low_memory=False)
print(f"Loaded: {df.shape}")

Loaded: (45466, 24)


## Step 2 — Drop the 6 corrupted/shifted rows

Identified in Prepare: a CSV export issue shifts columns on 3 pairs of adjacent rows, leaving
`id`, `budget`, or `popularity` unparseable as numbers.

In [2]:
bad_id = pd.to_numeric(df["id"], errors="coerce").isna()
bad_budget = pd.to_numeric(df["budget"], errors="coerce").isna()
bad_pop = pd.to_numeric(df["popularity"], errors="coerce").isna()
bad_any = bad_id | bad_budget | bad_pop
print(f"Dropping {bad_any.sum()} corrupted/shifted rows")
df = df.loc[~bad_any].copy()
print(f"After drop: {df.shape}")

Dropping 6 corrupted/shifted rows
After drop: (45460, 24)


## Step 3 — Coerce numeric types, drop duplicate `id` rows

In [3]:
df["id"] = pd.to_numeric(df["id"], errors="coerce").astype("Int64")
for col in ["budget", "revenue", "popularity", "runtime", "vote_average", "vote_count"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

before = len(df)
df = df.drop_duplicates(subset=["id"], keep="first")
print(f"Dropped {before - len(df)} duplicate-id rows (kept first occurrence)")
print(f"Shape: {df.shape}")

Dropped 30 duplicate-id rows (kept first occurrence)
Shape: (45430, 24)


## Step 4 — Parse `release_date`

In [4]:
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = df["release_date"].dt.year
df["release_month"] = df["release_date"].dt.month
print(f"Rows with unparseable release_date: {df["release_date"].isna().sum()}")

Rows with unparseable release_date: 84


## Step 5 — Parse `genres` (stringified list of dicts)

`ast.literal_eval` (not `eval`, for safety) turns `'[{'id': 16, 'name': 'Animation'}]'` into a real
Python list; we keep just the genre names.

In [5]:
def parse_genre_names(x):
    if pd.isna(x):
        return []
    try:
        parsed = ast.literal_eval(x)
        return [g["name"] for g in parsed if "name" in g]
    except (ValueError, SyntaxError):
        return []

df["genres_list"] = df["genres"].apply(parse_genre_names)
n_no_genre = (df["genres_list"].apply(len) == 0).sum()
print(f"Rows with zero parsed genres: {n_no_genre}")

Rows with zero parsed genres: 2442


## Step 6 — Treat `budget`/`revenue` of 0 as missing; compute `roi`/`profit`

Verified in Prepare: `0` means "not reported" on TMDB, not a literal $0. ROI/profit are only
computed where both `budget` and `revenue` exceed $1,000 — a low bar that filters out obvious
placeholder values without discarding genuine low-budget films.

In [6]:
df["budget_clean"] = df["budget"].where(df["budget"] > 0)
df["revenue_clean"] = df["revenue"].where(df["revenue"] > 0)
print(f"budget known (nonzero): {df["budget_clean"].notna().sum()}")
print(f"revenue known (nonzero): {df["revenue_clean"].notna().sum()}")

valid_financial = (df["budget_clean"] > 1000) & (df["revenue_clean"] > 1000)
df["profit"] = np.where(valid_financial, df["revenue_clean"] - df["budget_clean"], np.nan)
df["roi"] = np.where(valid_financial, (df["revenue_clean"] - df["budget_clean"]) / df["budget_clean"], np.nan)
print(f"Rows with valid roi/profit: {valid_financial.sum()}")
print(df.loc[valid_financial, "roi"].describe())

budget known (nonzero): 8880
revenue known (nonzero): 7398
Rows with valid roi/profit: 5307
count     5307.000000
mean         8.070325
std        189.568101
min         -0.999790
25%         -0.200266
50%          1.069137
75%          3.239814
max      12889.386667
Name: roi, dtype: float64


## Step 7 — Build the final movie-level table and the exploded movie x genre table

The movie-level table keeps `genres_list` intact (a Python list per row) for movie-level use; a
separate exploded table (one row per movie x genre) supports genre-level aggregation without
double-representing multi-genre movies in a single-value groupby.

In [7]:
keep_cols = [
    "id", "title", "release_date", "release_year", "release_month", "status",
    "genres_list", "runtime", "budget_clean", "revenue_clean", "profit", "roi",
    "vote_average", "vote_count", "popularity", "original_language", "adult", "video",
]
movies = df[keep_cols].rename(columns={"budget_clean": "budget", "revenue_clean": "revenue"})
print(f"Final movie-level table: {movies.shape}")

exploded = df[["id", "title", "genres_list"]].explode("genres_list").rename(columns={"genres_list": "genre"})
exploded = exploded.dropna(subset=["genre"])
exploded = exploded[exploded["genre"] != ""]
print(f"Exploded movie x genre table: {exploded.shape}, {exploded["genre"].nunique()} distinct genres")
print(exploded["genre"].value_counts().head(10))

Final movie-level table: (45430, 18)
Exploded movie x genre table: (91006, 3), 20 distinct genres
genre
Drama              20243
Comedy             13176
Thriller            7618
Romance             6730
Action              6590
Horror              4670
Crime               4304
Documentary         3930
Adventure           3490
Science Fiction     3042
Name: count, dtype: int64


## Step 8 — Save the processed datasets

In [8]:
movies_path = os.path.join(OUT_DIR, "movies_clean.parquet")
movies.to_parquet(movies_path, index=False)
size_kb = os.path.getsize(movies_path) / 1024
print(f"Saved movies_clean.parquet ({size_kb:.1f} KB), {movies.shape}")

genres_path = os.path.join(OUT_DIR, "movie_genres.parquet")
exploded.to_parquet(genres_path, index=False)
size_kb2 = os.path.getsize(genres_path) / 1024
print(f"Saved movie_genres.parquet ({size_kb2:.1f} KB), {exploded.shape}")

Saved movies_clean.parquet (2148.7 KB), (45430, 18)
Saved movie_genres.parquet (1162.3 KB), (91006, 3)


## Verification

The DuckDB SQL pipeline ([`sql/01_process_data.sql`](../sql/01_process_data.sql)) reproduces the
same final counts: 45,430 rows after the corrupted-row drop and dedup, 5,307 rows with a valid
`roi`, 8,880 rows with a known `budget`, 7,398 rows with a known `revenue`, and an identical
avg/median `roi` (8.0703 / 1.0691).

**One real cross-engine discrepancy worth documenting**: DuckDB's `read_csv` (in strict mode)
actually fails to parse the file at all — the same 6 corrupted rows contain a multi-line quoted
`overview` field that breaks DuckDB's column count validation (`Expected 24, found 10`). Loading
with `ignore_errors = true` makes DuckDB silently *skip* those 6 malformed rows at parse time
(landing on 45,460 loaded rows directly, with 0 further corrupted rows found by the
id/budget/popularity check) — while pandas' more lenient parser loads all 45,466 rows including
the corrupted ones, which are then explicitly filtered out by the id/budget/popularity validity
check. Different mechanisms, same 6 rows excluded, same final row count — a good illustration of
why cross-validating two engines is worth doing even when they "obviously" should agree.

## Summary

| Step | Result |
|---|---|
| Corrupted/shifted rows dropped | 6 |
| Duplicate-id rows dropped | 30 |
| Final movie-level rows | 45,430 |
| Rows with known budget (nonzero) | 8,880 |
| Rows with known revenue (nonzero) | 7,398 |
| Rows with valid roi/profit (both > $1,000) | 5,307 |
| Exploded movie x genre rows | 91,006 (20 distinct genres) |
| Cross-validated against DuckDB | Yes — see note above on the CSV-parsing discrepancy |